In [6]:
import torch 
import torch.nn as nn 
import numpy as np 

In [7]:
class PositionalEncodeing(nn.Module):
    def __init__(self,d_model,max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len,d_model)
        position = torch.arange(0,max_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0,d_model,2).float()*(-math.log(10000.0)/d.model))
        pe[:,0::2] = torch.sin(position * div_term)
        pe[:,1::2] = torch.cos(position*div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe',pe)
    def forward(self,x):
        x = x+self.pe[:,x.size(1)]
        return x

In [11]:

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)  # shape: (1, max_len, d_model)
        self.register_buffer('pe', pe)

    def forward(self, x):
        # x: (batch, seq_len, d_model)
        x = x + self.pe[:, :x.size(1)]
        return x


class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        self.q_linear = nn.Linear(d_model, d_model)
        self.k_linear = nn.Linear(d_model, d_model)
        self.v_linear = nn.Linear(d_model, d_model)
        self.out = nn.Linear(d_model, d_model)
    
    def forward(self, x, mask=None):
        batch_size, seq_len, d_model = x.size()
        
        Q = self.q_linear(x)
        K = self.k_linear(x)
        V = self.v_linear(x)
        
        Q = Q.view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1,2)  
        K = K.view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1,2)
        V = V.view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1,2)
        
        scores = torch.matmul(Q, K.transpose(-2,-1)) / math.sqrt(self.d_k)  
        
        if mask is not None:
            mask = mask.unsqueeze(1).unsqueeze(2)
            scores = scores.masked_fill(mask == 0, float('-inf'))
        
        attn = torch.softmax(scores, dim=-1)
        out = torch.matmul(attn, V)  
        
        out = out.transpose(1,2).contiguous().view(batch_size, seq_len, d_model)
        return self.out(out)


class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.relu = nn.ReLU()
    
    def forward(self, x):
        return self.linear2(self.relu(self.linear1(x)))


class EncoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.attn = MultiHeadSelfAttention(d_model, num_heads)
        self.ff = FeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, mask=None):
        x2 = self.attn(x, mask)
        x = self.norm1(x + self.dropout(x2))
        
        x2 = self.ff(x)
        x = self.norm2(x + self.dropout(x2))
        return x

class TransformerEncoder(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, d_ff, num_layers, max_len=512, dropout=0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_enc = PositionalEncoding(d_model, max_len)
        self.layers = nn.ModuleList([
            EncoderBlock(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)
        ])
    
    def forward(self, x, mask=None):
        x = self.embedding(x)  
        x = self.pos_enc(x)
        for layer in self.layers:
            x = layer(x, mask)
        return x  


batch_size = 2
seq_len = 5
vocab_size = 100
d_model = 16
num_heads = 4
d_ff = 64
num_layers = 2

x = torch.randint(0, vocab_size, (batch_size, seq_len))

mask = torch.ones(batch_size, seq_len)  

encoder = TransformerEncoder(vocab_size, d_model, num_heads, d_ff, num_layers)
out = encoder(x, mask)
print(out)
print(out.shape)  


tensor([[[ 0.2069, -0.2454,  1.2945,  0.8904, -2.2723,  0.2186,  0.2715,
           0.3710, -0.7595,  1.6134, -1.1595, -0.4810, -1.5042,  0.8432,
           0.2496,  0.4626],
         [ 0.5784,  0.7653, -1.8179,  1.4425, -1.0667, -1.8671, -0.1256,
           1.3692, -1.1434,  0.0828,  0.7855,  1.0428,  0.0647,  0.1078,
          -0.3556,  0.1372],
         [ 0.7980, -1.1111, -2.2747,  0.8225,  0.8985,  0.8765, -0.9795,
           1.3404,  1.0560, -0.2614,  0.1453,  0.3562, -1.5626, -0.2566,
           0.2946, -0.1420],
         [ 1.3693, -1.7259,  0.4906, -0.7129,  1.0567, -0.2171, -1.0564,
           2.0533, -1.1308,  0.0996,  0.6287,  0.5556, -1.1166, -0.4696,
          -0.3740,  0.5495],
         [ 0.5374,  1.8362,  0.0778, -0.2287, -1.1931,  1.0856, -1.1575,
           0.0408, -0.7791,  0.7901,  0.0250,  1.8427, -0.9854, -1.3656,
           0.3436, -0.8697]],

        [[-0.5904,  1.2523, -0.0122,  1.5577, -0.6235, -0.8767,  0.7519,
           0.2575, -0.4948,  0.4895, -0.4250, -1.6

In [12]:

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return x



class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        self.q_linear = nn.Linear(d_model, d_model)
        self.k_linear = nn.Linear(d_model, d_model)
        self.v_linear = nn.Linear(d_model, d_model)
        self.out = nn.Linear(d_model, d_model)
    
    def forward(self, x, mask=None):
        batch_size, seq_len, d_model = x.size()
        
        Q = self.q_linear(x)
        K = self.k_linear(x)
        V = self.v_linear(x)
        
        Q = Q.view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1,2)
        K = K.view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1,2)
        V = V.view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1,2)
        
        scores = torch.matmul(Q, K.transpose(-2,-1)) / math.sqrt(self.d_k) 
        
        causal_mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1).to(x.device)
        causal_mask = causal_mask.bool().unsqueeze(0).unsqueeze(1)  
        scores = scores.masked_fill(causal_mask, float('-inf'))
        
        if mask is not None:
            mask = mask.unsqueeze(1).unsqueeze(2)  
            scores = scores.masked_fill(mask == 0, float('-inf'))
        
        attn = torch.softmax(scores, dim=-1)
        out = torch.matmul(attn, V)
        out = out.transpose(1,2).contiguous().view(batch_size, seq_len, d_model)
        return self.out(out)


class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.relu = nn.ReLU()
    
    def forward(self, x):
        return self.linear2(self.relu(self.linear1(x)))

class DecoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.attn = MultiHeadSelfAttention(d_model, num_heads)
        self.ff = FeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, mask=None):
        x2 = self.attn(x, mask)
        x = self.norm1(x + self.dropout(x2))
        
        x2 = self.ff(x)
        x = self.norm2(x + self.dropout(x2))
        return x


class TransformerDecoder(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, d_ff, num_layers, max_len=512, dropout=0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_enc = PositionalEncoding(d_model, max_len)
        self.layers = nn.ModuleList([
            DecoderBlock(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)
        ])
        self.out = nn.Linear(d_model, vocab_size)  # final logits for each token
    
    def forward(self, x, mask=None):
        x = self.embedding(x)
        x = self.pos_enc(x)
        for layer in self.layers:
            x = layer(x, mask)
        return self.out(x)  


batch_size = 2
seq_len = 5
vocab_size = 100
d_model = 16
num_heads = 4
d_ff = 64
num_layers = 2

x = torch.randint(0, vocab_size, (batch_size, seq_len))
mask = torch.ones(batch_size, seq_len)  

decoder = TransformerDecoder(vocab_size, d_model, num_heads, d_ff, num_layers)
out = decoder(x, mask)
print(out)
print(out.shape)  


tensor([[[-8.0179e-01,  1.5054e-01,  1.6164e-01,  7.0645e-01,  1.1018e+00,
          -3.5918e-01, -8.4404e-01,  1.8024e+00,  4.6420e-02,  1.5330e-01,
           5.6459e-01, -4.1210e-01, -2.2648e-01, -3.7635e-01,  4.6060e-01,
          -3.5087e-01, -5.1632e-01, -7.2502e-01,  3.9736e-01,  7.3414e-01,
          -5.0603e-01,  4.7804e-01, -9.2447e-01, -9.2805e-02,  9.5526e-02,
          -2.4152e-02,  2.2605e-01, -5.3218e-02, -2.1666e-01, -2.7088e-01,
           9.3260e-01,  5.0059e-01, -3.8545e-01,  1.2195e-01,  3.5066e-01,
          -6.3052e-01,  7.1556e-01,  1.0376e+00,  4.5700e-01, -9.1105e-01,
          -6.7753e-01,  1.2515e+00,  1.7120e-01, -2.1012e-01,  5.0691e-02,
           1.1629e-01,  5.9708e-03, -1.5044e-01,  6.7980e-02,  1.1521e+00,
           1.8082e-01, -3.2221e-01, -3.7432e-01, -6.0827e-02,  9.0968e-01,
          -4.9906e-02, -4.1620e-01,  2.6586e-01,  2.5465e-01, -4.7635e-01,
          -1.2309e-01,  6.5793e-01, -1.2537e-01, -8.7438e-01,  7.3316e-01,
           1.6691e-01, -6